
# Optimization in Finance — HW2  
**Q1 (Part 2): Gurobi-Python Solution**

This notebook implements and solves the LP from Q1 (Part 1) using **Gurobi-Python**.  
We solve two instances by changing the revolving credit line rate:
- **Q1.2.a**: \( r_{CL} = 0.015 \) (1.5% per month)  
- **Q1.2.b**: \( r_{CL} = 0.025 \) (2.5% per month)

We then report optimal cash paths, credit-line usage, CP issuance, and final cash \(s_6\).


## Imports

In [ ]:

import math
import pandas as pd
from collections import defaultdict

try:
    import gurobipy as gp
    from gurobipy import GRB
except Exception as e:
    raise RuntimeError("This notebook requires Gurobi (gurobipy). Please ensure Gurobi is installed and licensed.\n"
                       "Install: pip install gurobipy (requires a valid Gurobi license).") from e



## Problem Data

We use the **compact** LP formulation (drop draw/repay variables; use the identity \(l_t - r_t = b_t - b_{t-1}\)).  
Interest in month \(t\) is charged on the **beginning-of-month** line balance \(b_{t-1}\).  
All debts are repaid by month 6 (i.e., \( b_6 = 0 \)).


In [ ]:

# Timeline
T = list(range(1, 7))  # months 1..6

# Operating cash flows (before financing)
CF = {1: 50, 2: -100, 3: -150, 4: 200, 5: -300, 6: 400}

# Credit line
CAP = 100  # capacity
# rCL will be set per run

# Commercial Papers: list of dicts for clarity
# Fields: j, issue, mature, coupon
CPs = [
    {"j": 1, "issue": 1, "mature": 2, "coupon": 0.02},
    {"j": 2, "issue": 1, "mature": 4, "coupon": 0.025},
    {"j": 3, "issue": 2, "mature": 5, "coupon": 0.035},
    {"j": 4, "issue": 3, "mature": 6, "coupon": 0.02},
    {"j": 5, "issue": 4, "mature": 6, "coupon": 0.03},
]
J = [cp["j"] for cp in CPs]

# Helper lookups
issue_at = {cp["j"]: cp["issue"] for cp in CPs}
mature_at = {cp["j"]: cp["mature"] for cp in CPs}
coupon = {cp["j"]: cp["coupon"] for cp in CPs}



## Model Builder (Compact Form)

We create a function that builds and solves the LP for a given \( r_{CL} \).  
**Variables**:  
- \( s_t \ge 0 \): end-of-month cash  
- \( b_t \ge 0 \): end-of-month credit-line balance  
- \( y_j \ge 0 \): face of CP \(j\)

**Constraints**:  
- Capacity: \( 0 \le b_t \le 100 \)  
- Terminal: \( b_6 = 0 \)  
- Cash recursion (for each \(t\in\{1,\dots,6\}\)):  
\[
s_t = s_{t-1} + \mathrm{CF}_t + \sum_{j: i_j=t} y_j - \sum_{j: m_j=t} (1+c_j)y_j + (b_t - b_{t-1}) - r_{CL} b_{t-1}.
\]
with \( s_0 = b_0 = 0 \).


In [ ]:

def build_and_solve(rCL, verbose=False):
    m = gp.Model("CashFinancingLP")
    if not verbose:
        m.Params.OutputFlag = 0  # silence solver output by default

    # Decision variables
    s = m.addVars(T, name="s", lb=0.0)    # end-of-month cash
    b = m.addVars(T, name="b", lb=0.0)    # end-of-month CL balance
    y = m.addVars(J, name="y", lb=0.0)    # CP issuance

    # Capacity and terminal constraints
    for t in T:
        m.addConstr(b[t] <= CAP, name=f"cap[{t}]")
    m.addConstr(b[6] == 0, name="terminal")

    # Cash-balance recursion
    for t in T:
        inflow_cp = gp.quicksum(y[j] for j in J if issue_at[j] == t)
        outflow_cp = gp.quicksum((1.0 + coupon[j]) * y[j] for j in J if mature_at[j] == t)
        b_prev = 0.0 if t == 1 else b[t-1]
        s_prev = 0.0 if t == 1 else s[t-1]
        m.addConstr(
            s[t] == s_prev + CF[t] + inflow_cp - outflow_cp + (b[t] - b_prev) - rCL * b_prev,
            name=f"cash[{t}]"
        )

    # Objective: maximize s_6
    m.setObjective(s[6], GRB.MAXIMIZE)

    # Optimize
    m.optimize()

    if m.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Optimization did not reach optimality. Status: {m.Status}")

    # Extract results
    s_val = {t: s[t].X for t in T}
    b_val = {t: b[t].X for t in T}
    y_val = {j: y[j].X for j in J}

    # Build a period-by-period accounting table
    rows = []
    for t in T:
        inflow_cp_t = sum(y_val[j] for j in J if issue_at[j] == t)
        outflow_cp_t = sum((1.0 + coupon[j]) * y_val[j] for j in J if mature_at[j] == t)
        b_prev = 0.0 if t == 1 else b_val[t-1]
        net_draw = b_val[t] - b_prev
        interest = rCL * b_prev
        rows.append({
            "t": t,
            "CF": CF[t],
            "CP_inflow": inflow_cp_t,
            "CP_outflow": outflow_cp_t,
            "Line_net_draw": net_draw,
            "Interest": interest,
            "Cash_end": s_val[t],
            "Line_end": b_val[t],
        })
    df = pd.DataFrame(rows).set_index("t")

    # Basic sanity checks
    tol = 1e-6
    checks = {
        "all_cash_nonnegative": all(s_val[t] >= -tol for t in T),
        "capacity_never_exceeded": all(b_val[t] <= CAP + tol for t in T),
        "terminal_balance_zero": abs(b_val[6]) <= tol,
    }

    # Compact summary
    summary = {
        "rCL": rCL,
        "s6": s_val[6],
        "CP_issued": {j: y_val[j] for j in J if y_val[j] > tol},
        "Line_usage": {
            "max_balance": max(b_val.values()),
            "draw_months": [t for t in T if (b_val[t] - (0.0 if t==1 else b_val[t-1])) > tol],
            "repay_months": [t for t in T if (b_val[t] - (0.0 if t==1 else b_val[t-1])) < -tol],
        },
        "checks": checks,
    }

    return m, s_val, b_val, y_val, df, summary



## Solve the Two Instances
We run the model for \( r_{CL}=0.015 \) and \( r_{CL}=0.025 \) and display:
- the period-by-period accounting table, and  
- a compact summary of decisions.


In [ ]:

# Q1.2.a: rCL = 1.5% per month
m1, s1, b1, y1, df1, summary1 = build_and_solve(rCL=0.015, verbose=False)
display(df1.style.format({c: "{:.3f}" for c in df1.columns if c != "CF"}))
print("\nSummary (rCL=0.015):")
print(summary1)


In [ ]:

# Q1.2.b: rCL = 2.5% per month
m2, s2, b2, y2, df2, summary2 = build_and_solve(rCL=0.025, verbose=False)
display(df2.style.format({c: "{:.3f}" for c in df2.columns if c != "CF"}))
print("\nSummary (rCL=0.025):")
print(summary2)



## Sanity Checks
We verify: \( s_t \ge 0 \) (no bankruptcy), \( b_t \le 100 \) (capacity), and \( b_6 = 0 \) (terminal).


In [ ]:

def print_checks(tag, summary):
    print(f"Checks for {tag}:")
    for k, v in summary["checks"].items():
        print(f"  {k}: {v}")
    print()

print_checks("rCL=0.015", summary1)
print_checks("rCL=0.025", summary2)



## Save Outputs and Export Notebook (Instructions)

- The tables `df1` and `df2` are saved as CSV for convenience.
- To export this notebook to **PDF**, use: **File → Save and Export Notebook As… → PDF** (or use `nbconvert` if configured).  
We'll later embed the exported PDF in the HW2 LaTeX.


In [ ]:

df1.to_csv("q1_part2_results_r015.csv", index=True)
df2.to_csv("q1_part2_results_r025.csv", index=True)
print("Saved: q1_part2_results_r015.csv, q1_part2_results_r025.csv")
